# PAP vs. PMP: circuit quality against computational cost

Compares the two IPE discovery modalities on **IOI**, across three models, every search
strategy, and both positional settings, and reports what each configuration *costs* against
what it *buys*.

| | |
|---|---|
| **PAP** — `PathAttributionPatching` | prunes on the first-order gradient estimate `<msg, grad>` |
| **PMP** — `PathMessagePatching` | prunes on the exact patched score `evaluate_path` |
| **PMP batched** | PMP with `batch_heads` (and `batch_positions` when positional): score an attention block whole, split only the winners into heads/positions |

PMP batched is included because it is PMP as you would actually deploy it — 4-10x faster in
`experiments/resource_consumption/` — while plain PMP is the strict algorithmic-parity
comparison against PAP, which has no batching equivalent. Reading both separates *"is the
gradient approximation worth it"* from *"is exact patching affordable"*.

**Tree search is deliberately excluded here**; it is a separate axis, covered by
`experiments/faithfulness_completeness.ipynb` and `experiments/tree_vs_path.py`.

## What is measured

* **Faithfulness** — Wang et al. (2023, §3), by knockout: run the model with every component
  *not* in the circuit mean-ablated over the ABC distribution, read off the IOI logit
  difference `F(C) = E[logit(IO) - logit(S)]`, and normalise so `1.0` is the full model and
  `0.0` the empty circuit. Implemented in `experiments/knockout.py`, which is that notebook's
  harness parameterised by the model rather than reading `N_LAYERS`/`N_HEADS` from globals.
* **Nodes retained** — `|components in C| / (n_layers x (n_heads + 1))`, plus the head-only and
  MLP-only fractions. This is the circuit's size, and the axis the table should really be read
  along: PAP and PMP score on different scales, so a shared `min_contribution` does *not* make
  two rows comparable — a shared circuit size does.
* **Time** — wall clock of the search call alone, excluding model load, data and evaluation.
* **Memory** — peak `torch.cuda` allocation during the search, and the delta over what was
  already resident (weights + activation caches) when it started, which is the search's own
  footprint.

Two scopes are reported for faithfulness, and the gap between them is itself informative:
`all` ablates everything outside the circuit, MLPs included; `attention` counts only the
circuit's heads as the circuit and leaves every MLP intact. Ablating the MLP sublayers damages
the model on its own, so under `all` a circuit that found few MLPs scores near zero however
good its heads are. `attention - all` is the faithfulness carried by the MLPs a search missed.

## Design notes

**Every search now takes a wall-clock budget.** Only `*_BestFirstSearch` had `max_time`;
`Threshold` and `LimitedLevelWidth` could run unbounded, which is untenable for an 8B model.
`src/ipe/graph_search.py` gained a `_Deadline` helper and a `max_time` argument on all six
searches. It defaults to `None` (unlimited) on the four that had no budget and stays `300` on
the two that did, so nothing that ran before changes behaviour. When the budget expires the
search returns the paths completed so far — valid, just smaller, because every path is scored
in isolation — and sets `root.timed_out`, which each row carries as `timed_out`.

**A truncated row measures the budget, not the algorithm.** Any row with `timed_out=True`
reports `max_time` as its time, so it says "this is the circuit this method reaches in an
hour", not "this is what this method costs". Both readings are useful; don't mix them.

**Thresholds are not comparable across models.** `min_contribution` lives on the scale of the
`logit_difference` metric, which differs per model. Compare at matched circuit size.

**Numerics.** GPT-2 and Qwen run in fp32; Llama-3-8B runs in bf16 because 8B x 4 bytes does not
fit a 24GB card. Its faithfulness numbers carry bf16 rounding that the other two do not.

In [1]:
import json, os, subprocess, sys, textwrap
from collections import Counter

import pandas as pd

EXPERIMENTS_DIR = os.getcwd()                       # this notebook lives in experiments/
REPO_ROOT = os.path.dirname(EXPERIMENTS_DIR)
for p in (os.path.join(REPO_ROOT, "src"), REPO_ROOT, EXPERIMENTS_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)                       # `ipe` is used from src/, not pip-installed

OUT_DIR = os.path.join(EXPERIMENTS_DIR, "pap_vs_pmp_performance_computational_cost")
WORKER = os.path.join(EXPERIMENTS_DIR, "pap_vs_pmp_worker.py")
os.makedirs(OUT_DIR, exist_ok=True)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
print(f"outputs -> {OUT_DIR}")

outputs -> /home/nicolobrunello/Documents/Projects/IPE-LatentCircuitIdentification/experiments/pap_vs_pmp_performance_computational_cost


## 1. Configuration

`MODELS` is the sweep. Each entry becomes one subprocess invocation of
`experiments/pap_vs_pmp_worker.py`, which loads the model **once** and runs that model's whole
grid, caching each cell to its own JSON so a re-run resumes rather than restarts.

`MAX_TIME` is the per-search budget. It is the single knob that bounds the sweep: worst case is
`MAX_TIME x 42 cells x 3 models`.

In [ ]:
MAX_TIME = 3600.0          # wall-clock budget per search, seconds

# The grid definition and the model roster live in `pap_vs_pmp_grid`, which the standalone
# driver `experiments/run_pap_vs_pmp.py` imports too, so the notebook and the script always run
# the same sweep. That module is free of torch, so this cell needs no GPU.
from pap_vs_pmp_grid import MODELS, MODALITIES, THRESHOLDS, TOP_NS, MAX_WIDTHS, build_grid

GRID = build_grid(MAX_TIME)
print(f"modalities  {list(MODALITIES)}")
print(f"positional  [False, True]")
print(f"thresholds  {THRESHOLDS}    top_n {TOP_NS}    max_width {MAX_WIDTHS}")
print(f"models      {[m for m, _ in MODELS]}")
print(f"\n{len(GRID)} cells per model x {len(MODELS)} models = {len(GRID)*len(MODELS)} searches")
print(f"worst case  {len(GRID)*len(MODELS)*MAX_TIME/3600:.0f} GPU-hours if every search times out")
print(f"            (GPT-2 and Qwen mostly finish well inside the budget; Llama-3-8B mostly will not)")
pd.DataFrame([{ "run_id": c["run_id"], "modality": c["modality"], "positional": c["positional"],
                "strategy": c["strategy"],
                "params": {k: v for k, v in c["params"].items()
                           if k not in ("max_time", "include_negative")} } for c in GRID]).head(14)

## 2. Smoke test

Runs the six `max_width=100` cells on GPT-2 (three modalities x two positional settings) with a 60s budget, into a throwaway directory, to check the whole
path end to end — search, serialisation, knockout, scoring — before committing real GPU time.
Expect a couple of minutes, most of it model download and the knockout baselines.

In [3]:
SMOKE_DIR = os.path.join(OUT_DIR, "_smoke")

def run_worker(model, flags, out_dir=OUT_DIR, max_time=MAX_TIME, extra=()):
    """Invoke the worker for one model, streaming its output into the notebook."""
    cmd = [sys.executable, WORKER, "--model", model, "--out-dir", out_dir,
           "--max-time", str(max_time), *flags, *extra]
    env = {**os.environ, "PYTHONPATH": os.path.join(REPO_ROOT, "src") + os.pathsep
                                       + os.environ.get("PYTHONPATH", "")}
    print("$ " + " ".join(cmd) + "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env, cwd=EXPERIMENTS_DIR)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\n[exit {proc.returncode}]")
    return proc.returncode

run_worker("gpt2-small", ["--dtype", "float32", "--eval-minibatch", "16"],
           out_dir=SMOKE_DIR, max_time=60,
           extra=["--only", "LimitedLevelWidth_w100", "--eval-size", "16", "--search-batch", "3"])

$ /home/nicolobrunello/miniconda/envs/nlp/bin/python /home/nicolobrunello/Documents/Projects/IPE-LatentCircuitIdentification/experiments/pap_vs_pmp_worker.py --model gpt2-small --out-dir /home/nicolobrunello/Documents/Projects/IPE-LatentCircuitIdentification/experiments/pap_vs_pmp_performance_computational_cost/_smoke --max-time 60 --dtype float32 --eval-minibatch 16 --only LimitedLevelWidth_w100 --eval-size 16 --search-batch 3

loading gpt2-small (float32) on cuda ...
Loaded pretrained model gpt2-small into HookedTransformer
  12 layers x 12 heads
  IOI at length 16: 3 search + 16 eval
  weights: 622 MB

search phase: 12 of 42 cells to run (budget 60s each, worst case 0.2h)

[1/12] pap_nopos_LimitedLevelWidth_w100

100%|██████████| 1/1 [00:00<00:00,  8.35it/s]

100%|██████████| 100/100 [00:00<00:00, 246.32it/s]

100%|██████████| 100/100 [00:00<00:00, 236.72it/s]

100%|██████████| 100/100 [00:00<00:00, 255.83it/s]

100%|██████████| 100/100 [00:00<00:00, 285.44it/s]

100%|██████████| 10

0

## 3. The sweep

**This is the expensive cell, and for a full run you probably want the script instead.**
`experiments/run_pap_vs_pmp.py` does exactly what this section does but survives a lost kernel,
which matters when the sweep is measured in days:

```bash
tmux new -s ipe
python experiments/run_pap_vs_pmp.py status     # what is already cached; no GPU needed
python experiments/run_pap_vs_pmp.py run        # runs, or resumes
# Ctrl-b d to detach, `tmux attach -t ipe` to come back
```

Both paths share `pap_vs_pmp_grid` and `pap_vs_pmp_tables`, run the same worker, and write to
the same cache, so you can move between them freely — run the sweep in tmux and read the tables
here, or start in the notebook and finish in the script.

Either way it is resumable: one subprocess per model, every cell cached to its own JSON, and a
cell counts as done only if its file exists *and* parses. Writes are atomic, so an interrupt
mid-write cannot leave a file that resume would trust.

Before launching Llama-3-8B, note that it needs ~16GB of weights in bf16 on a 24GB card, so the
GPU has to be otherwise idle, and the checkpoint is gated on HuggingFace (`huggingface-cli
login`). If it OOMs, the subprocess dies without taking this kernel with it, and the GPT-2 and
Qwen results already on disk are unaffected.

In [ ]:
for model, flags in MODELS:
    print("=" * 100)
    print(f"  {model}")
    print("=" * 100, flush=True)
    rc = run_worker(model, flags)
    if rc != 0:
        print(f"!! {model} exited {rc}; continuing with the next model "
              f"(cached cells are kept, re-run this cell to resume)")

  gpt2-small
$ /home/nicolobrunello/miniconda/envs/nlp/bin/python /home/nicolobrunello/Documents/Projects/IPE-LatentCircuitIdentification/experiments/pap_vs_pmp_worker.py --model gpt2-small --out-dir /home/nicolobrunello/Documents/Projects/IPE-LatentCircuitIdentification/experiments/pap_vs_pmp_performance_computational_cost --max-time 3600.0 --dtype float32 --eval-minibatch 16

loading gpt2-small (float32) on cuda ...
Loaded pretrained model gpt2-small into HookedTransformer
  12 layers x 12 heads
  IOI at length 16: 5 search + 64 eval
  weights: 622 MB

search phase: 42 of 42 cells to run (budget 3600s each, worst case 42.0h)

[1/42] pap_nopos_LimitedLevelWidth_w100

100%|██████████| 1/1 [00:00<00:00,  9.56it/s]

100%|██████████| 100/100 [00:00<00:00, 231.72it/s]

100%|██████████| 100/100 [00:00<00:00, 227.77it/s]

100%|██████████| 100/100 [00:00<00:00, 239.96it/s]

100%|██████████| 100/100 [00:00<00:00, 275.75it/s]

100%|██████████| 100/100 [00:00<00:00, 293.61it/s]

100%|██████████|

## 4. Results

Every scored circuit is one JSON file under `pap_vs_pmp_performance_computational_cost/scores/`.
This section reads whatever is on disk, so it runs without a GPU and works on a partial sweep.

In [ ]:
from pap_vs_pmp_tables import (load_scores, build_table, modality_summary,
                               matched_size, save_tables, ROUNDING)

SCORES = load_scores(OUT_DIR)
if SCORES.empty:
    print("no scored circuits yet — run section 3 (or the script) through its eval phase")
else:
    ok = (SCORES["status"] == "ok") & (SCORES["eval_status"] == "ok")
    print(f"{len(SCORES)} runs on disk   |   {ok.sum()} scored   "
          f"|   {(~ok).sum()} failed   |   {SCORES['timed_out'].fillna(False).sum()} timed out")
    print("\nby model:", dict(Counter(SCORES["model"])))
    failed = SCORES[~ok]
    if len(failed):
        print("\nfailures:")
        for _, r in failed.iterrows():
            print(f"  {r['model']:30s} {r['run_id']:42s} {str(r.get('error'))[:90]}")

### 4.1 The table

One row per (model, modality, positional, strategy, hyperparameter). This is the deliverable.

`faith_all` and `faith_attn` are the two knockout scopes, normalised (1.0 = full model,
0.0 = empty circuit). `%nodes` / `%heads` / `%mlps` are the fractions of the model retained.
`time_s` and `peak_mb` are the search's own cost; `TO` marks a row that hit the budget and is
therefore a lower bound on what the method would eventually find.

In [ ]:
TABLE = build_table(SCORES)
if not TABLE.empty:
    display(TABLE.round(ROUNDING))
else:
    print("nothing scored yet")

### 4.2 Modality comparison

The headline question, one row per (model, positional, modality): what each modality achieves
and what it costs, aggregated over that modality's 7 hyperparameter cells. `best_faith_all` is
its best circuit; `median_time_s` is the typical cost of one search.

In [ ]:
MODALITY = modality_summary(TABLE)
if not MODALITY.empty:
    display(MODALITY.round({"best_faith_all": 3, "best_faith_attn": 3, "median_pct_nodes": 2,
                            "median_time_s": 1, "total_time_s": 0, "median_peak_mb": 0}))

### 4.3 Matched circuit size

The comparison that actually holds up. PAP and PMP rank candidates on different scales, so a
shared `min_contribution` says nothing; a shared circuit size does. For each model and each size
band, this is the best faithfulness each modality reaches and what it paid — so a cell where PAP
matches PMP's faithfulness at a fraction of the time is the case for the approximation, and one
where it does not is the case against.

In [ ]:
MATCHED = matched_size(TABLE)
if not MATCHED.empty:
    display(MATCHED.round(3))

### 4.4 Save

Writes the flat table and both summaries next to the raw runs, as CSV (for further analysis)
and Markdown (to paste into the thesis).

In [ ]:
if not TABLE.empty:
    written = save_tables({"table_full": TABLE, "summary_modality": MODALITY,
                           "summary_matched_size": MATCHED}, OUT_DIR)
    print("\n".join(written) or "(nothing written)")
    if not any(p.endswith(".md") for p in written):
        print("(no .md files — pip install tabulate for Markdown output)")
else:
    print("nothing to save yet")